In [2]:
import boto3
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter
from sagemaker.estimator import Estimator
from sagemaker import image_uris
from sagemaker.serializers import CSVSerializer

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
url = "https://frauddateset.s3.ap-southeast-2.amazonaws.com/merged_dataset.csv"
d1 = pd.read_csv(url)

d1.head(3)

,Transaction.Date,Transaction.Amount,Customer.Age,Is.Fraudulent,Account.Age.Days,Transaction.Hour,source,browser,sex,Payment.Method,Product.Category,Quantity,Device.Used,Address.Match
0,2024-02-12 10:05:21,145.98,29,0,172,10,Ads,IE,F,credit card,home & garden,3,mobile,1
1,2024-01-25 22:24:06,677.62,40,0,250,22,Direct,FireFox,M,credit card,clothing,3,desktop,1
2,2024-03-26 20:32:44,798.63,40,0,118,20,Ads,Chrome,M,PayPal,clothing,3,mobile,1


In [11]:
print(d1['Is.Fraudulent'].value_counts(normalize=True))  # Confirms 95/5
print(d1.info())  # Data types OK?

# Encode categoricals (for ML)
d1 = pd.get_dummies(d1, columns=['Payment.Method', 'Product.Category', 'Device.Used'])


Is.Fraudulent
0    0.92799
1    0.07201
Name: proportion, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Transaction.Date    300000 non-null  object 
 1   Transaction.Amount  300000 non-null  float64
 2   Customer.Age        300000 non-null  int64  
 3   Is.Fraudulent       300000 non-null  int64  
 4   Account.Age.Days    300000 non-null  int64  
 5   Transaction.Hour    300000 non-null  int64  
 6   source              300000 non-null  object 
 7   browser             300000 non-null  object 
 8   sex                 300000 non-null  object 
 9   Payment.Method      300000 non-null  object 
 10  Product.Category    300000 non-null  object 
 11  Quantity            300000 non-null  int64  
 12  Device.Used         300000 non-null  object 
 13  Address.Match       300000 non-null  int64  
dtypes: float64(

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1️⃣ Separate majority and minority classes
df_majority = d1[d1['Is.Fraudulent'] == 0]
df_minority = d1[d1['Is.Fraudulent'] == 1]

print("Original class distribution:")
print(d1['Is.Fraudulent'].value_counts())

# 2️⃣ Undersample majority class
df_majority_downsampled = resample(
    df_majority,
    replace=False,            # sample without replacement
    n_samples=int(len(df_minority)*5),  # 5x minority
    random_state=42
)

# 3️⃣ Optional: Slightly oversample minority
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=int(len(df_minority)*2),  # double minority
    random_state=42
)

# 4️⃣ Combine sampled data
d1_balanced = pd.concat([df_majority_downsampled, df_minority_upsampled])

# Shuffle
d1_balanced = d1_balanced.sample(frac=1, random_state=42)

print("Balanced dataset class distribution:")
print(d1_balanced['Is.Fraudulent'].value_counts())

# --- Start of fix ---
# Convert 'Transaction.Date' to datetime objects and extract numerical features
d1_balanced['Transaction.Date'] = pd.to_datetime(d1_balanced['Transaction.Date'], errors='coerce')
d1_balanced = d1_balanced.dropna(subset=['Transaction.Date']) # Drop rows where date conversion failed
d1_balanced['Trans_Year'] = d1_balanced['Transaction.Date'].dt.year
d1_balanced['Trans_Month'] = d1_balanced['Transaction.Date'].dt.month
d1_balanced['Trans_Day'] = d1_balanced['Transaction.Date'].dt.day
d1_balanced['Trans_DayOfWeek'] = d1_balanced['Transaction.Date'].dt.dayofweek
d1_balanced.drop(columns=['Transaction.Date'], inplace=True)

# One-hot encode categorical columns if not already done in d1
# This cell's previous execution state didn't include comprehensive one-hot encoding here,
# assuming 'd1' might not have all categoricals encoded before this step.
cat_cols = ['source', 'browser', 'sex', 'Payment.Method', 'Product.Category', 'Device.Used']
cat_cols_to_encode = [col for col in cat_cols if col in d1_balanced.columns]
d1_balanced = pd.get_dummies(d1_balanced, columns=cat_cols_to_encode, drop_first=True)
# --- End of fix ---

# 5️⃣ Split features and target
X = d1_balanced.drop('Is.Fraudulent', axis=1)
y = d1_balanced['Is.Fraudulent']

# 6️⃣ Encode categorical columns (This step is no longer needed as columns were already one-hot encoded)
# cat_cols = ['source', 'browser', 'sex', 'Payment.Method', 'Product.Category', 'Device.Used']
# for col in cat_cols:
#     X[col] = X[col].astype('category').cat.codes

# 7️⃣ Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 8️⃣ Train RandomForest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 9️⃣ Evaluate
y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


# Predict probabilities
y_prob = model.predict_proba(X_test)[:,1]

# Lower threshold to 0.3 or 0.25
threshold = 0.2478
y_pred_thresh = (y_prob > threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report
print(f"Confusion Matrix (threshold={threshold}):")
print(confusion_matrix(y_test, y_pred_thresh))

print(f"\nClassification Report (threshold={threshold}):")
print(classification_report(y_test, y_pred_thresh, digits=4))


Original class distribution:
Is.Fraudulent
0    278397
1     21603
Name: count, dtype: int64
Balanced dataset class distribution:
Is.Fraudulent
0    108015
1     43206
Name: count, dtype: int64
Confusion Matrix:
[[21307   296]
 [ 4258  4383]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8334    0.9863    0.9035     21603
           1     0.9367    0.5072    0.6581      8641

    accuracy                         0.8494     30244
   macro avg     0.8851    0.7468    0.7808     30244
weighted avg     0.8630    0.8494    0.8334     30244

Confusion Matrix (threshold=0.2478):
[[20161  1442]
 [ 2768  5873]]

Classification Report (threshold=0.2478):
              precision    recall  f1-score   support

           0     0.8793    0.9333    0.9055     21603
           1     0.8029    0.6797    0.7361      8641

    accuracy                         0.8608     30244
   macro avg     0.8411    0.8065    0.8208     30244
weighted avg     0.8574

In [15]:
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

# Read raw data
url = "https://frauddateset.s3.ap-southeast-2.amazonaws.com/merged_dataset.csv" 
df = pd.read_csv(url)

# Split by class
df_majority = df[df['Is.Fraudulent']==0]
df_minority = df[df['Is.Fraudulent']==1]

# Downsample majority and upsample minority
df_majority_downsampled = resample(df_majority,
                                   replace=False,
                                   n_samples=len(df_minority)*5,
                                   random_state=42)

df_minority_upsampled = resample(df_minority,
                                 replace=True,
                                 n_samples=len(df_minority)*2,
                                 random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_majority_downsampled, df_minority_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42)

# Convert date to numeric features
df_balanced['Transaction.Date'] = pd.to_datetime(df_balanced['Transaction.Date'], errors='coerce')
df_balanced = df_balanced.dropna(subset=['Transaction.Date'])
df_balanced['Trans_Year'] = df_balanced['Transaction.Date'].dt.year
df_balanced['Trans_Month'] = df_balanced['Transaction.Date'].dt.month
df_balanced['Trans_Day'] = df_balanced['Transaction.Date'].dt.day
df_balanced['Trans_DayOfWeek'] = df_balanced['Transaction.Date'].dt.dayofweek
df_balanced.drop(columns=['Transaction.Date'], inplace=True)

# One-hot encode categorical columns
cat_cols = ['source', 'browser', 'sex', 'Payment.Method', 'Product.Category', 'Device.Used']
df_balanced = pd.get_dummies(df_balanced, columns=cat_cols, drop_first=True)

# Split features/target
X = df_balanced.drop('Is.Fraudulent', axis=1)
y = df_balanced['Is.Fraudulent']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Save to CSV for SageMaker
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)


In [5]:
import sagemaker

session = sagemaker.Session()
bucket = session.default_bucket()

trainpath = session.upload_data(
    path="train_preprocessed.csv",
    bucket=bucket,
    
)

testpath = session.upload_data(
    path="test_preprocessed.csv",
    bucket=bucket,
   
)

print(trainpath)
print(testpath)

s3://sagemaker-ap-southeast-2-212208750989/data/train_preprocessed.csv
s3://sagemaker-ap-southeast-2-212208750989/data/test_preprocessed.csv


In [8]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

role = get_execution_role()

sklearn_estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    base_job_name="rf-custom-sklearn",
    hyperparameters={"n_estimators": 100, "random_state": 42}
)

sklearn_estimator.fit({
    "train": "s3://sagemaker-ap-southeast-2-212208750989/data/train_preprocessed.csv",
    "test":  "s3://sagemaker-ap-southeast-2-212208750989/data/test_preprocessed.csv"
})


INFO:sagemaker:Creating training-job with name: rf-custom-sklearn-2026-02-14-03-30-43-184


2026-02-14 03:30:48 Starting - Starting the training job...
2026-02-14 03:31:04 Starting - Preparing the instances for training...
2026-02-14 03:31:45 Downloading - Downloading the training image......
2026-02-14 03:32:46 Training - Training image download completed. Training in progress.../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-14 03:32:51,006 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-02-14 03:32:51,010 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-02-14 03:32:51,013 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-02-14 03:32:51,029 sagemaker_skle

In [9]:
MODEL_DATA = "s3://sagemaker-ap-southeast-2-212208750989/rf-custom-sklearn-2026-02-02-04-04-53-175/output/model.tar.gz"

from sagemaker.sklearn.model import SKLearnModel

sklearn_model = SKLearnModel(
    model_data=MODEL_DATA,
    role=role,
    entry_point="script.py",
    framework_version="1.2-1"
)



In [ ]:
from sagemaker.sklearn.model import SKLearnModel

MODEL_DATA = sklearn_estimator.model_data  # ✅ always use this

sklearn_model = SKLearnModel(
    model_data=MODEL_DATA,
    role=role,
    entry_point="script.py",   # only if you have custom inference
    framework_version="1.2-1"
)


In [12]:
predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)


INFO:sagemaker:Creating model with name: rf-custom-sklearn-2026-02-14-03-36-03-278
INFO:sagemaker:Creating endpoint-config with name rf-custom-sklearn-2026-02-14-03-36-03-278
INFO:sagemaker:Creating endpoint with name rf-custom-sklearn-2026-02-14-03-36-03-278


------!

In [13]:
import pandas as pd
import numpy as np

# Take one row from test set
sample = X_test.iloc[0:1]

result = predictor.predict(sample.values.tolist())

print("Prediction:", result)
print(X_train.shape[1])

Prediction: [1]
26


In [16]:

print(X_train.columns)
sample = X_train.iloc[0].values.tolist()
print(sample)
sample = X_train.iloc[[0]]   # double brackets keep it 2D

# Predict
prediction = model.predict(sample)

print("Prediction:", prediction)


Index(['Transaction.Amount', 'Customer.Age', 'Account.Age.Days',
       'Transaction.Hour', 'Quantity', 'Address.Match', 'Trans_Year',
       'Trans_Month', 'Trans_Day', 'Trans_DayOfWeek', 'source_Direct',
       'source_SEO', 'browser_FireFox', 'browser_IE', 'browser_Opera',
       'browser_Safari', 'sex_M', 'Payment.Method_bank transfer',
       'Payment.Method_credit card', 'Payment.Method_debit card',
       'Product.Category_electronics', 'Product.Category_health & beauty',
       'Product.Category_home & garden', 'Product.Category_toys & games',
       'Device.Used_mobile', 'Device.Used_tablet'],
      dtype='object')
[305.64, 47, 111, 2, 2, 1, 2024, 2, 2, 4, False, True, False, False, False, True, False, False, False, False, True, False, False, False, False, True]
Prediction: [0]
